In [2]:
from bs4 import BeautifulSoup as soup
import requests

### Grabbing the URL, Set, and Packs from the main website

First, we find the set, and the correspond packs for those sets.

In [3]:
from bs4.element import Tag

main_page = "https://pkmncards.com/sets/"

html    = requests.get(main_page)
parsed  = soup(html.content)
sets    = parsed.find_all("h2")

sets_and_grouped_packs : dict[str, Tag] = {s.text:s.find_next_sibling() for s in sets}

Parsing out the `Tag` into individual packs

In [4]:
sets_and_split_packs : dict[str, list[tuple[str, str]]] = { }

# Parsing out the URL and name of each individual pack
for s, grouped_packs in sets_and_grouped_packs.items():
	raw_packs = grouped_packs.find_all("a")
	parsed_packs = []
	for p in raw_packs:
		sets_and_split_packs.setdefault(s, []).append((p.text, p.get("href")))

sets_and_split_packs

{'Mega Evolution': [('Chaos Rising (CRI)',
   'https://pkmncards.com/set/chaos-rising/'),
  ('Perfect Order (POR)', 'https://pkmncards.com/set/perfect-order/'),
  ('Ascended Heroes (ASC)', 'https://pkmncards.com/set/ascended-heroes/'),
  ('Phantasmal Flames (PFL)', 'https://pkmncards.com/set/phantasmal-flames/'),
  ('Mega Evolution Energy (MEE)', 'https://pkmncards.com/set/mee/'),
  ('Mega Evolution (MEG)', 'https://pkmncards.com/set/mega-evolution/'),
  ('Mega Evolution Promos (MEP)',
   'https://pkmncards.com/set/mega-evolution-promos/')],
 'Scarlet & Violet': [('White Flare (WHT)',
   'https://pkmncards.com/set/white-flare/'),
  ('Scarlet & Violet Energy (SVE)',
   'https://pkmncards.com/set/scarlet-violet-energy/'),
  ('Black Bolt (BLK)', 'https://pkmncards.com/set/black-bolt/'),
  ('Destined Rivals (DRI)', 'https://pkmncards.com/set/destined-rivals/'),
  ('Journey Together (JTG)', 'https://pkmncards.com/set/journey-together/'),
  ('Prismatic Evolutions (PRE)',
   'https://pkmncard

### Fetching card information for each pack

In [7]:
import pandas as pd
from tqdm.notebook import tqdm

cards = []

for s, packs in tqdm(sets_and_split_packs.items(), desc="Sets", leave=False):
	for pack_name, pack_url in tqdm(packs, desc="Packs in a set", leave=False):
		url    = pack_url + "?sort=date&ord=auto&display=full"
		html   = requests.get(url).content
		parsed = soup(html)

		card_elems = parsed.find_all(class_="type-pkmn_card")
		for c in tqdm(card_elems, desc="Cards in a pack", leave=False):
			name          = c.find("span", title="Name").text
			image_url     = c.find("img", class_="card-image").get("src")
			collector_num = c.find("span", class_="number-out-of").text
			rarity        = c.find("span", class_="rarity").text
			release_date  = c.find("span", title="Date Released").text

			# Fetching price
			price_box = c.find("div", class_="card-pricing available")
			if price_box:
				l = price_box.find("li", title="Lowest Price").span.text
				m = price_box.find("li", title="Market Price").span.text
				h = price_box.find("li", title="Highest Price").span.text
			else:
				l = None
				m = None
				h = None
			
			cards.append({
				"set": s,
				"pack": pack_name,
				"name": name,
				"image_url": image_url,
				"rarity": rarity,
				"collector_number": collector_num,
				"release_date": release_date,
				"lowest_price": l,
				"market_price": m,
				"highest_price": h
			})

Sets:   0%|          | 0/20 [00:00<?, ?it/s]

Packs in a set:   0%|          | 0/7 [00:00<?, ?it/s]

Cards in a pack:   0%|          | 0/122 [00:00<?, ?it/s]

Cards in a pack:   0%|          | 0/124 [00:00<?, ?it/s]

Cards in a pack:   0%|          | 0/295 [00:00<?, ?it/s]

...

Cards in a pack:   0%|          | 0/34 [00:00<?, ?it/s]

Cards in a pack:   0%|          | 0/34 [00:00<?, ?it/s]

Cards in a pack:   0%|          | 0/34 [00:00<?, ?it/s]

Packs in a set:   0%|          | 0/1 [00:00<?, ?it/s]

Cards in a pack:   0%|          | 0/94 [00:00<?, ?it/s]

In [20]:
import uuid

cards_df = pd.DataFrame(cards)

def string_num_to_float(snum: str | float) -> float:
	if isinstance(snum, str):
		return float(snum.replace(",", ""))
	return snum



# Applying this function to all number columns
cols = ["lowest_price", "market_price", "highest_price"]
cards_df[cols] = cards_df[cols].apply(lambda col: col.map(string_num_to_float))

# Setting uuid
cards_df["card_id"] = [uuid.uuid4() for s in range(len(cards_df))]

# Removing '#' from collector_number
cards_df["collector_number"] = cards_df["collector_number"].apply(lambda s: s[1:])

# Removing '↘' from date
cards_df["release_date"] = cards_df["release_date"].apply(lambda s: s[1:])

# Reordering colummns
cols = ["card_id", "set", "pack", "name", "image_url", "rarity", "collector_number", "release_date", "lowest_price", "market_price", "highest_price"]
cards_df = cards_df[cols]

cards_df.sort_values(by="lowest_price", ascending=False)

,card_id,set,pack,name,image_url,rarity,collector_number,release_date,lowest_price,market_price,highest_price
17229,fbff42bb-fbc9-47eb-baea-4157f47ded9c,EX,Power Keepers (PK),Vaporeon Star,https://pkmncards.com/wp-content/uploads/vapor...,Rare Holo Star,102/108,"Feb 14, 2007",99998.99,900.99,99999.00
19344,6f1608b9-6faa-4533-9635-9566ccecd691,E-Card,Aquapolis (AQ),Nidoking,https://pkmncards.com/wp-content/uploads/nidok...,Rare Secret,150/147,"Jan 15, 2003",20000.99,600.00,20000.99
17335,08cf220e-fbf3-4ca0-b128-b2a0aa3a1776,EX,Dragon Frontiers (DF),Charizard Star,https://pkmncards.com/wp-content/uploads/chari...,Rare Holo Star,100/101,"Nov 8, 2006",20000.00,4000.00,39500.00
22551,e5dc8190-2252-4f89-8b5b-6ecd641783fc,Other,Box Topper,Charizard,https://pkmncards.com/wp-content/uploads/p09-b...,Rare Holo,9/12,"Jan 15, 2003",12000.00,0.00,12000.00
15157,b6564c0c-21b7-488c-8b65-1310ecdec97f,HeartGold & SoulSilver,Call of Legends (CL),Rayquaza,https://pkmncards.com/wp-content/uploads/en_US...,Rare Holo,SL10/95,"Feb 9, 2011",9999.99,543.19,27494.90
...,...,...,...,...,...,...,...,...,...,...,...
22700,0a88069c-f630-46b5-b17e-afd3b36215c2,Other,Victory Medals,Victory Medal,https://pkmncards.com/wp-content/uploads/victo...,No Rarity,o Number,"Sep 24, 2010",NaN,NaN,NaN
22701,578893f0-6fb8-49e2-b927-be91caa5c342,Other,Victory Medals,Victory Medal,https://pkmncards.com/wp-content/uploads/victo...,No Rarity,o Number,"May 30, 2011",NaN,NaN,NaN
22704,bfc1d90c-8b41-4d8f-b0b7-50c1267d3ee6,Misc.,Pokémon Trading Card Game Classic—Venusaur (CLV),Venusaur,https://pkmncards.com/wp-content/uploads/clv_e...,No Rarity,003/034,"Nov 17, 2023",NaN,NaN,NaN
22777,6ec53676-dac5-4301-9161-3ea5d468e1c5,Misc.,Pokémon Trading Card Game Classic—Blastoise (CLB),Lapras,https://pkmncards.com/wp-content/uploads/clb_e...,No Rarity,008/034,"Nov 17, 2023",NaN,NaN,NaN


### Saving Data

In [26]:
# saving dataframe
cards_df.to_csv("all_pokemon_cards_v1.csv", index=False)